In [1]:
import sys
import os

In [ ]:
# from helpers.dataframeBuilder import get_df_neigbours_answer_for_all_itemIDs
from sklearn.preprocessing import OneHotEncoder
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
df = pd.read_csv('../csv/preprocessed_data.csv')
df_norm = pd.read_csv('../csv/preprocessed_data_norm.csv')
df_ips = pd.read_csv('../csv/preprocessed_data_ips.csv')

In [ ]:
df.set_index(['TestKey'])

In [ ]:
X = df.select_dtypes(include=[np.number]).values

In [ ]:
# Vervang NaN-waarden door de mediaan van elke kolom
from sklearn.impute import SimpleImputer
imputer = SimpleImputer(strategy='median')
X = imputer.fit_transform(X)

In [ ]:
X = X.astype(np.float64)

In [ ]:
print(X.shape)

# Modellen trainen

In [ ]:
# voor elke model dezelfde contamination
contamination= 0.05

In [ ]:
def predict_model(model, df, name):
    df_answers=df
    
    model.fit(X)
    
    df_answers['anomaly_score'] = model.decision_scores_
    df_answers['is_anomaly'] = model.predict(X)

    df_answers = df_answers.reset_index()
    df_answers = df_answers[['TestKey','anomaly_score', 'is_anomaly']]
    
    print(f"Aantal gedetecteerde anomalieën: {df_answers['is_anomaly'].sum()}")

    df_answers.set_index('TestKey', inplace=True)
    df_answers.to_csv(f"../csv/{name}.csv")
    df_answers.reset_index(inplace=True)
    
    return df_answers

## ECOD

In [ ]:
from pyod.models.ecod import ECOD

ecod = ECOD(contamination=contamination)

In [ ]:
df_ecod = predict_model(ecod, df, "ecod")

In [ ]:
df_ecod[df_ecod['is_anomaly'] == 1].head()

## Isolation forest

In [ ]:
from pyod.models.iforest import IForest

i_forest = IForest(contamination=contamination, n_estimators=100, max_samples='auto')

In [ ]:
df_iForest = predict_model(i_forest, df, 'i_forest')

## LOF

In [ ]:
from pyod.models.lof import LOF

lof = LOF(contamination=contamination)

In [ ]:
df_lof = predict_model(lof, df, 'lof')

## COPOD 

In [ ]:
from pyod.models.copod import COPOD

copod = COPOD(contamination=contamination)

In [ ]:
df_copod = predict_model(copod, df,'copod')

## ABOD

In [ ]:
from pyod.models.abod import ABOD

abod = ABOD(contamination=contamination, method='fast', n_neighbors=10)

In [ ]:
df_abod = predict_model(abod, df, 'abod')

In [ ]:
df_abod[df_abod['is_anomaly'] == 1].head()

## Overeenkomsten

### functie voor analyse

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib_venn import venn3, venn2

def compare_anomaly_models(*model_dfs, id_column='TestKey'):
    """
    Vergelijkt anomaliedetecties van meerdere modellen.
    
    :param model_dfs: Een lijst van tuples, elk met (dataframe, model_name)
    :param id_column: De naam van de kolom die als identifier wordt gebruikt
    :return: Een DataFrame met gecombineerde resultaten
    """
    # Initialiseer de gecombineerde DataFrame met de ID-kolom
    df_combined = pd.DataFrame({id_column: model_dfs[0][0][id_column]})
    
    # Voeg resultaten van elk model toe
    model_names = []
    for df, model_name in model_dfs:
        model_names.append(model_name)
        df_combined = df_combined.merge(
            df[[id_column, 'is_anomaly']],
            on=id_column,
            suffixes=('', f'_{model_name}')
        )
        df_combined = df_combined.rename(columns={'is_anomaly': f'{model_name}_anomaly'})
    
    # Tel het aantal anomaliedetecties per datapunt
    anomaly_columns = [col for col in df_combined.columns if col.endswith('_anomaly')]
    df_combined['anomaly_count'] = df_combined[anomaly_columns].sum(axis=1)
    
    # Analyseer de overeenkomsten
    print("Verdeling van anomaliedetecties:")
    print(df_combined['anomaly_count'].value_counts().sort_index())
    
    # # Visualiseer de overeenkomsten
    plt.figure(figsize=(10, 6))
    df_combined['anomaly_count'].value_counts().sort_index().plot(kind='bar')
    plt.title('Aantal datapunten gedetecteerd als anomalie door x modellen')
    plt.xlabel('Aantal modellen')
    plt.ylabel('Aantal datapunten')
    plt.show()
    
    # Bekijk consensus anomalieën
    consensus_anomalies = df_combined[df_combined['anomaly_count'] == len(model_dfs)]
    print(f"\nAantal datapunten gedetecteerd als anomalie door alle {len(model_dfs)} modellen:", 
          len(consensus_anomalies))
    # print("\nVoorbeelden van consensus anomalieën:")
    # print(consensus_anomalies.head())
    
    # Vergelijk modellen paarsgewijs
    print("\nPaarsgewijze vergelijking van modellen:")
    for i in range(len(model_names)):
        for j in range(i+1, len(model_names)):
            agreement = (df_combined[f'{model_names[i]}_anomaly'] == df_combined[f'{model_names[j]}_anomaly']).mean()
            print(f"Overeenkomst tussen {model_names[i].upper()} en {model_names[j].upper()}: {agreement:.2%}")
    
    # Visualiseer de overlap van anomalieën met een Venn diagram alleen als er 3 modellen zijn
    if len(model_names) == 3:
        plt.figure(figsize=(8, 7))
        venn3([set(df_combined[df_combined[f'{model}_anomaly'] == 1][id_column]) for model in model_names],
              set_labels=model_names)
        plt.title("Overlap van gedetecteerde anomalieën")
        plt.show()
    else:
        print("\nVenn diagram wordt alleen gemaakt voor exact 3 modellen.")
    
    return df_combined

### uitvoeren

In [ ]:
df_combined = compare_anomaly_models(
    (df_ecod, 'ecod'),
    (df_iForest, 'i_forest'),
    (df_lof, 'lof'),
    (df_copod, 'copod'),
    (df_abod, 'abod'),
)

In [ ]:
df_combined

In [ ]:
df_combined.to_csv('../csv/comparison.csv', index=False)

# SOUD

In [ ]:
# from pyod.models.suod import SUOD
# from pyod.models.lof import LOF
# from pyod.models.iforest import IForest
# from pyod.models.copod import COPOD
# from pyod.models.knn import KNN
# from pyod.models.hbos import HBOS

In [ ]:
# base_estimators = [
#     IForest(n_estimators=100, max_samples='auto', contamination=contamination, random_state=42),
#     # LOF(n_neighbors=20, contamination=contamination),
#     ECOD(contamination=contamination),
#     COPOD(contamination=contamination)
# ]

In [ ]:
# # Initialiseer SUOD
# suod = SUOD(base_estimators=base_estimators, 
#             n_jobs=2,  # aantal parallelle jobs
#             combination='average',  # methode om scores te combineren
#             verbose=False)

In [ ]:
# def data_generator(X, batch_size=1000):
#     for i in range(0, len(X), batch_size):
#         yield X[i:i + batch_size]

In [ ]:
# # Functie om SUOD te trainen en voorspellingen te maken met batches
# def process_batches(X, batch_size=1000):
#     anomaly_scores = []
    
#     # Train SUOD op batches
#     for batch in data_generator(X, batch_size):
#         suod.fit(batch)  # Train op de huidige batch
#         batch_scores = suod.decision_function(batch)  # Verkrijg anomalie scores voor de batch
#         anomaly_scores.extend(batch_scores)  # Voeg scores toe aan de lijst
    
#     return np.array(anomaly_scores)

In [ ]:
# # Verwerk de data in batches en verkrijg anomalie scores
# anomaly_scores = process_batches(X, batch_size=1000)

# # Bepaal anomalieën op basis van de scores (bijvoorbeeld top 5% als anomalieën)
# threshold = np.percentile(anomaly_scores, 95)  # Drempelwaarde instellen
# is_anomaly = anomaly_scores > threshold

In [ ]:
# df_suod = pd.DataFrame({
#     'TestKey': np.arange(1, len(X) + 1),  # Voeg een identificatiekolom toe (bijv. FCATestKey)
#     'anomaly_score': anomaly_scores,
#     'is_anomaly': is_anomaly.astype(int)  # Zet booleans om naar integers (0 of 1)
# })

In [ ]:
# print(f"Aantal gedetecteerde anomalieën: {df_suod['is_anomaly'].sum()}")
# print(df_suod.head())  # Toon de eerste paar resultaten

In [ ]:
# df_suod = df_suod.set_index('TestKey')
# df_suod.to_csv('../csv/suod_baq.csv')